In [9]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

In [10]:
# Пример данных
data = pd.DataFrame({
    'Date': pd.to_datetime(['2024-12-25', '2024-11-15', '2024-07-01', '2024-01-01']),
    'Product': ['Milk', 'Candy', 'Juice', 'Bread'],
    'Discount': [1, 0, 1, 0],
    'Holiday': [1, 0, 0, 1],
    'Sales': [120, 80, 150, 130]
})

In [11]:
data['Month'] = data['Date'].dt.month
data = pd.get_dummies(data, columns=['Product'], drop_first=True)

In [12]:
scaler = StandardScaler()
X = scaler.fit_transform(data[['Discount', 'Holiday', 'Month'] + list(data.columns[5:])])
y = data['Sales']

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=13)

In [14]:
# 1. Ансамблевые модели
# Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=13)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

# Gradient Boosting
gb_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=13)
gb_model.fit(X_train, y_train)
gb_pred = gb_model.predict(X_test)

# 2. Полносвязная нейронная сеть (FCNN)
fcnn_model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='linear')
])

C:\Users\santa\PycharmProjects\Diploma\venv\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [15]:
# Компиляция модели
fcnn_model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

# Обучение модели
fcnn_model.fit(X_train, y_train, epochs=50, batch_size=8, validation_split=0.2)

# Прогнозирование нейронной сети
fcnn_pred = fcnn_model.predict(X_test)

# 4. Прогнозирование запаса на складе после закупки
# Используем прогнозы модели для расчета уровня запаса
initial_stock = 500  # пример начального уровня запаса
predicted_demand = np.mean(fcnn_pred)  # используем среднее прогнозируемое значение для оценки спроса
stock_after_purchase = initial_stock - predicted_demand

Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - loss: 15620.3770 - mae: 124.8940 - val_loss: 22591.9434 - val_mae: 150.3062
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - loss: 15600.5947 - mae: 124.8141 - val_loss: 22584.9180 - val_mae: 150.2828
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - loss: 15580.8965 - mae: 124.7345 - val_loss: 22577.7305 - val_mae: 150.2589
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - loss: 15561.4033 - mae: 124.6558 - val_loss: 22570.7305 - val_mae: 150.2356
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - loss: 15541.9766 - mae: 124.5773 - val_loss: 22563.8750 - val_mae: 150.2128
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - loss: 15523.1123 - mae: 124.5011 - val_loss: 22557.1562 - val_mae: 150.1904
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - loss: 15504.6035 - mae: 124.4263 - val_loss: 22550.1758 - val_mae: 150.1672
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - loss: 15487.0908 - mae: 124.3555 - val_loss: 22543.2520 - val_

1. Ансамбли (Random Forest и Gradient Boosting) используют данные о скидках и праздниках для прогноза продаж. Результаты этих моделей позволяют выявить более глубокие зависимости.
2. Полносвязная нейронная сеть (FCNN) принимает те же входные данные и создает нелинейную модель, предсказывая объем продаж на основе скидок и праздников.
3. Запас на складе рассчитывается с учетом предсказанного спроса.

In [16]:
# Пример вывода
output = {
    "Product_Name": "Milk",
    "Stock_After_Purchase": stock_after_purchase,
    "Predicted_Demand": predicted_demand
}

print("Random Forest Prediction:", rf_pred)
print("Gradient Boosting Prediction:", gb_pred)
print("FCNN Prediction:", fcnn_pred.flatten())
print(output)

Random Forest Prediction: [134.9]
Gradient Boosting Prediction: [136.32942068]
FCNN Prediction: [0.9970502]
{'Product_Name': 'Milk', 'Stock_After_Purchase': np.float32(499.00296), 'Predicted_Demand': np.float32(0.9970502)}
